<a href="https://colab.research.google.com/github/Miranita-ar/Skripsi-Gojek-App-Review/blob/main/Code/(XE)_Skripsi_IndoBERT_(Analisis_Hasil_XAI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Persiapan Lingkungan

Notebook ini digunakan untuk melakukan analisis hasil Explainable Artificial Intelligence (XAI) berdasarkan output dari notebook XC (SHAP Analysis) dan notebook XD (LIME Analysis). Hasil akhir notebook ini berupa visualisasi SHAP dan LIME yang siap digunakan pada BAB 4 skripsi.


## 1.1 Instal Library

In [ ]:
# =====================================================
# CELL 1 : INSTALL LIBRARY
# =====================================================

!pip -q install shap
!pip -q install lime
!pip -q install transformers
!pip -q install sentencepiece
!pip -q install accelerate
!pip -q install safetensors

print("="*60)
print("INSTALL LIBRARY")
print("="*60)
print()
print("✅ Seluruh library berhasil diinstall.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 13.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INSTALL LIBRARY

✅ Seluruh library berhasil diinstall.


## 1.2 Import Library

In [ ]:
# =====================================================
# CELL 2 : IMPORT LIBRARY
# =====================================================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

import torch
import shap

warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300

print("="*60)
print("IMPORT LIBRARY")
print("="*60)
print()
print("✅ Library berhasil diimport.")

IMPORT LIBRARY

✅ Library berhasil diimport.


## 1.3 Mount Google Drive

In [ ]:
# =====================================================
# CELL 3 : MOUNT GOOGLE DRIVE
# =====================================================

from google.colab import drive

drive.mount("/content/drive")

print("="*60)
print("MOUNT GOOGLE DRIVE")
print("="*60)
print()
print("✅ Google Drive berhasil di-mount.")

Mounted at /content/drive
MOUNT GOOGLE DRIVE

✅ Google Drive berhasil di-mount.


## 1.4 Import Utility Function

In [ ]:
# =====================================================
# CELL 4 : IMPORT UTILITY FUNCTION
# =====================================================

def print_header(title):
    print("\n" + "="*60)
    print(title)
    print("="*60)

def print_success(message):
    print(f"✅ {message}")

def print_info(message):
    print(f"ℹ️ {message}")

def print_warning(message):
    print(f"⚠️ {message}")

print_header("IMPORT UTILITY FUNCTION")

print()
print_success("Utility Function berhasil dibuat.")


IMPORT UTILITY FUNCTION

✅ Utility Function berhasil dibuat.


## 1.5 Konfigurasi Path

In [ ]:
# =====================================================
# CELL 5 : KONFIGURASI PATH
# =====================================================

PATHS = {

    "PROJECT":
    "/content/drive/MyDrive/Skripsi_IndoBERT",

    "XC":
    "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis",

    "XD":
    "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis",

    "XE":
    "/content/drive/MyDrive/Skripsi_IndoBERT/xai/XE_Analisis_Hasil_XAI"

}

print_header("KONFIGURASI PATH")

print()

for key, value in PATHS.items():
    print_info(f"{key:<8}: {value}")

print()

print_success("Seluruh path berhasil dikonfigurasi.")


KONFIGURASI PATH

ℹ️ PROJECT : /content/drive/MyDrive/Skripsi_IndoBERT
ℹ️ XC      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XC_SHAP_Analysis
ℹ️ XD      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XD_LIME_Analysis
ℹ️ XE      : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XE_Analisis_Hasil_XAI

✅ Seluruh path berhasil dikonfigurasi.


## 1.6 Validasi Path

In [ ]:
# =====================================================
# CELL 6 : VALIDASI PATH
# =====================================================

print_header("VALIDASI PATH")

print()

for key, value in PATHS.items():

    if os.path.exists(value):

        print_success(f"{key} tersedia")

    else:

        os.makedirs(value, exist_ok=True)

        print_info(f"{key} dibuat")

print()
print_success("Validasi Path selesai.")


VALIDASI PATH

✅ PROJECT tersedia
✅ XC tersedia
✅ XD tersedia
✅ XE tersedia

✅ Validasi Path selesai.


## 1.7 Konfigurasi Dictionary

In [ ]:
# =====================================================
# CELL 7 : KONFIGURASI DICTIONARY
# =====================================================

SENTIMENT_LABEL = {

    0: "Negatif",

    1: "Netral",

    2: "Positif"

}

CASE_LABEL = {

    "Benar_Positif":

        "Prediksi Benar Positif",

    "Benar_Netral":

        "Prediksi Benar Netral",

    "Benar_Negatif":

        "Prediksi Benar Negatif",

    "Salah_Netral_Pos":

        "Netral → Positif",

    "Salah_Netral_Neg":

        "Netral → Negatif",

    "Uncertain":

        "Ketidakpastian Tinggi"

}

print_header("KONFIGURASI DICTIONARY")

print()

print_info(f"Jumlah Sentimen : {len(SENTIMENT_LABEL)}")

print_info(f"Jumlah Kasus : {len(CASE_LABEL)}")

print()

print_success("Dictionary berhasil dibuat.")


KONFIGURASI DICTIONARY

ℹ️ Jumlah Sentimen : 3
ℹ️ Jumlah Kasus : 6

✅ Dictionary berhasil dibuat.


## 1.8 Validasi Dictionary

In [ ]:
# =====================================================
# CELL 8 : VALIDASI DICTIONARY
# =====================================================

print_header("VALIDASI DICTIONARY")

display(pd.DataFrame({

    "Kode":

        list(SENTIMENT_LABEL.keys()),

    "Sentimen":

        list(SENTIMENT_LABEL.values())

}))

print()

display(pd.DataFrame({

    "Kasus":

        list(CASE_LABEL.keys()),

    "Keterangan":

        list(CASE_LABEL.values())

}))

print()

print_success("Dictionary berhasil divalidasi.")


VALIDASI DICTIONARY


,Kode,Sentimen
0,0,Negatif
1,1,Netral
2,2,Positif


,Kasus,Keterangan
0,Benar_Positif,Prediksi Benar Positif
1,Benar_Netral,Prediksi Benar Netral
2,Benar_Negatif,Prediksi Benar Negatif
3,Salah_Netral_Pos,Netral → Positif
4,Salah_Netral_Neg,Netral → Negatif
5,Uncertain,Ketidakpastian Tinggi



✅ Dictionary berhasil divalidasi.


# BAB 2 Memuat Output Explainable Artificial Intelligence

Notebook ini memuat seluruh hasil Explainable Artificial Intelligence (XAI) yang telah dihasilkan pada notebook XC (SHAP Analysis) dan notebook XD (LIME Analysis). Seluruh output tersebut akan digunakan sebagai dasar untuk membangun visualisasi akhir serta analisis pada BAB 4 skripsi.


## 2.1 Load Output XC

In [ ]:
# =====================================================
# CELL 9 : LOAD OUTPUT XC
# =====================================================

print_header("LOAD OUTPUT XC")

XC_ROOT = PATHS["XC"]

XC_CSV = os.path.join(
    XC_ROOT,
    "csv"
)

XC_GLOBAL = os.path.join(
    XC_ROOT,
    "global_analysis"
)

XC_LOCAL = os.path.join(
    XC_ROOT,
    "local_metadata"
)

overall_shap_csv = os.path.join(
    XC_CSV,
    "overall_shap_analysis.csv"
)

if not os.path.exists(overall_shap_csv):

    raise FileNotFoundError(
        overall_shap_csv
    )

overall_shap_df = pd.read_csv(
    overall_shap_csv
)

print()

print_info(
    f"Overall SHAP : {len(overall_shap_df)} Token"
)

print()

print_success(
    "Output XC berhasil dimuat."
)


LOAD OUTPUT XC

ℹ️ Overall SHAP : 20 Token

✅ Output XC berhasil dimuat.


## 2.2 Validasi Output XC

In [ ]:
# =====================================================
# CELL 10 : VALIDASI OUTPUT XC
# =====================================================

print_header("VALIDASI OUTPUT XC")

summary_xc = pd.DataFrame({

    "Komponen":[

        "Overall SHAP CSV",

        "Jumlah Token",

        "Folder XC"

    ],

    "Nilai":[

        os.path.exists(overall_shap_csv),

        len(overall_shap_df),

        os.path.exists(XC_ROOT)

    ]

})

display(summary_xc)

print()

print_success(
    "Output XC berhasil divalidasi."
)


VALIDASI OUTPUT XC


,Komponen,Nilai
0,Overall SHAP CSV,True
1,Jumlah Token,20
2,Folder XC,True



✅ Output XC berhasil divalidasi.


## 2.3 Load Output XD

In [ ]:
# =====================================================
# CELL 11 : LOAD OUTPUT XD
# =====================================================

print_header("LOAD OUTPUT XD")

XD_ROOT = PATHS["XD"]

XD_CSV = os.path.join(
    XD_ROOT,
    "csv"
)

XD_LOCAL = os.path.join(
    XD_ROOT,
    "local_metadata"
)

overall_lime_csv = os.path.join(
    XD_CSV,
    "overall_lime_analysis.csv"
)

if not os.path.exists(overall_lime_csv):

    raise FileNotFoundError(
        overall_lime_csv
    )

overall_lime_df = pd.read_csv(
    overall_lime_csv
)

print()

print_info(
    f"Overall LIME : {len(overall_lime_df)} Token"
)

print()

print_success(
    "Output XD berhasil dimuat."
)


LOAD OUTPUT XD

ℹ️ Overall LIME : 20 Token

✅ Output XD berhasil dimuat.


## 2.4 Validasi Output XD

In [ ]:
# =====================================================
# CELL 12 : VALIDASI OUTPUT XD
# =====================================================

print_header("VALIDASI OUTPUT XD")

summary_xd = pd.DataFrame({

    "Komponen":[

        "Overall LIME CSV",

        "Jumlah Token",

        "Folder XD"

    ],

    "Nilai":[

        os.path.exists(overall_lime_csv),

        len(overall_lime_df),

        os.path.exists(XD_ROOT)

    ]

})

display(summary_xd)

print()

print_success(
    "Output XD berhasil divalidasi."
)


VALIDASI OUTPUT XD


,Komponen,Nilai
0,Overall LIME CSV,True
1,Jumlah Token,20
2,Folder XD,True



✅ Output XD berhasil divalidasi.


## 2.5 Sinkronisasi Data SHAP dan LIME

In [ ]:
# =====================================================
# CELL 13 : SINKRONISASI TOP TOKEN SHAP DAN LIME
# =====================================================

print_header("SINKRONISASI TOP TOKEN SHAP DAN LIME")

top_token_shap = len(overall_shap_df)

top_token_lime = len(overall_lime_df)

common_token = len(
    set(overall_shap_df["Token"]).intersection(
        set(overall_lime_df["Token"])
    )
)

comparison_summary = {

    "top_token_shap": top_token_shap,

    "top_token_lime": top_token_lime,

    "common_token": common_token

}

print()

print_info(
    f"Top Token SHAP : {top_token_shap}"
)

print_info(
    f"Top Token LIME : {top_token_lime}"
)

print_info(
    f"Common Top Token : {common_token}"
)

print()

print_success(
    "Sinkronisasi Top Token berhasil."
)


SINKRONISASI TOP TOKEN SHAP DAN LIME

ℹ️ Top Token SHAP : 20
ℹ️ Top Token LIME : 20
ℹ️ Common Top Token : 8

✅ Sinkronisasi Top Token berhasil.


## 2.6 Validasi Sinkronisasi Data

In [ ]:
# =====================================================
# CELL 14 : VALIDASI SINKRONISASI TOP TOKEN
# =====================================================

print_header("VALIDASI SINKRONISASI TOP TOKEN")

summary = pd.DataFrame({

    "Komponen":[

        "Top Token SHAP",

        "Top Token LIME",

        "Common Top Token"

    ],

    "Jumlah":[

        comparison_summary["top_token_shap"],

        comparison_summary["top_token_lime"],

        comparison_summary["common_token"]

    ]

})

display(summary)

print()

print_success(
    "Sinkronisasi Top Token berhasil divalidasi."
)


VALIDASI SINKRONISASI TOP TOKEN


,Komponen,Jumlah
0,Top Token SHAP,20
1,Top Token LIME,20
2,Common Top Token,8



✅ Sinkronisasi Top Token berhasil divalidasi.


# BAB 3 Persiapan Analisis 30 Sampel XAI

Bab ini memuat 30 sampel ulasan yang telah dipilih pada notebook XB sebagai objek analisis Explainable Artificial Intelligence (XAI). Dataset tersebut akan digunakan sebagai acuan utama dalam visualisasi SHAP dan LIME pada bab berikutnya.

## 3.1 Load Dataset XAI

In [ ]:
# =====================================================
# CELL 15 : LOAD DATASET XAI
# =====================================================

print_header("LOAD DATASET XAI")

SELECTED_DATASET_PATH = os.path.join(

    PATHS["PROJECT"],

    "xai",

    "XB_Persiapan_XAI",

    "csv",

    "selected_reviews_xai.csv"

)

if not os.path.exists(SELECTED_DATASET_PATH):

    raise FileNotFoundError(

        SELECTED_DATASET_PATH

    )

selected_reviews_df = pd.read_csv(

    SELECTED_DATASET_PATH

)

print()

print_info(

    f"Dataset : {SELECTED_DATASET_PATH}"

)

print_info(

    f"Jumlah Sampel : {len(selected_reviews_df)}"

)

print()

print_success(

    "Dataset XAI berhasil dimuat."

)


LOAD DATASET XAI

ℹ️ Dataset : /content/drive/MyDrive/Skripsi_IndoBERT/xai/XB_Persiapan_XAI/csv/selected_reviews_xai.csv
ℹ️ Jumlah Sampel : 30

✅ Dataset XAI berhasil dimuat.


## 3.2 Validasi Dataset XAI

In [ ]:
# =====================================================
# CELL 16 : VALIDASI DATASET XAI
# =====================================================

print_header("VALIDASI DATASET XAI")

display(

    selected_reviews_df.head()

)

print()

print_info(

    f"Jumlah Baris : {len(selected_reviews_df)}"

)

print_info(

    f"Jumlah Kolom : {len(selected_reviews_df.columns)}"

)

print()

display(

    pd.DataFrame({

        "Kolom":

            selected_reviews_df.columns

    })

)

print()

print_success(

    "Dataset XAI berhasil divalidasi."

)


VALIDASI DATASET XAI


,analysis_order,sample_id,text,actual_label,actual_sentiment,predicted_label,predicted_sentiment,prob_negatif,prob_netral,prob_positif,confidence,correct,prediction_case,confidence_level,selection_reason,sample_group,explain_status
0,1,XAI_001,sangat membantu dan mudah,2,Positif,2,Positif,0.000885,0.000943,0.998172,0.998172,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
1,2,XAI_002,mudah dan sangat membantu sekali,2,Positif,2,Positif,0.000879,0.000953,0.998167,0.998167,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
2,3,XAI_003,sangat membantu sekali,2,Positif,2,Positif,0.000898,0.000940,0.998162,0.998162,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
3,4,XAI_004,baik dan sangat membantu,2,Positif,2,Positif,0.000917,0.000924,0.998159,0.998159,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending
4,5,XAI_005,sangat membantu sekali dalam segala aktivitas,2,Positif,2,Positif,0.000891,0.000954,0.998155,0.998155,True,Benar_Positif,Tinggi,Prediksi benar kelas Positif,High Confidence,Pending



ℹ️ Jumlah Baris : 30
ℹ️ Jumlah Kolom : 17



,Kolom
0,analysis_order
1,sample_id
2,text
3,actual_label
4,actual_sentiment
5,predicted_label
6,predicted_sentiment
7,prob_negatif
8,prob_netral
9,prob_positif



✅ Dataset XAI berhasil divalidasi.


## 3.3 Persiapan Struktur Visualisasi

In [ ]:
# =====================================================
# CELL 17 : PERSIAPAN STRUKTUR VISUALISASI
# =====================================================

print_header("PERSIAPAN STRUKTUR VISUALISASI")

TOTAL_SAMPLE = len(

    selected_reviews_df

)

VISUALIZATION_CONFIG = {

    "total_sample":

        TOTAL_SAMPLE,

    "correct_prediction":

        15,

    "misclassification":

        10,

    "high_uncertainty":

        5

}

print()

for key, value in VISUALIZATION_CONFIG.items():

    print_info(

        f"{key:<22}: {value}"

    )

print()

print_success(

    "Struktur visualisasi berhasil dibuat."

)


PERSIAPAN STRUKTUR VISUALISASI

ℹ️ total_sample          : 30
ℹ️ correct_prediction    : 15
ℹ️ misclassification     : 10
ℹ️ high_uncertainty      : 5

✅ Struktur visualisasi berhasil dibuat.


## 3.4 Validasi Struktur Visualisasi

In [ ]:
# =====================================================
# CELL 18 : VALIDASI STRUKTUR VISUALISASI
# =====================================================

print_header("VALIDASI STRUKTUR VISUALISASI")

summary = pd.DataFrame({

    "Komponen":[

        "Total Sampel",

        "Prediksi Benar",

        "Misclassification",

        "High Uncertainty"

    ],

    "Jumlah":[

        VISUALIZATION_CONFIG["total_sample"],

        VISUALIZATION_CONFIG["correct_prediction"],

        VISUALIZATION_CONFIG["misclassification"],

        VISUALIZATION_CONFIG["high_uncertainty"]

    ]

})

display(summary)

print()

if VISUALIZATION_CONFIG["total_sample"] == 30:

    print_success(

        "Struktur visualisasi sesuai dengan desain penelitian."

    )

else:

    print_warning(

        "Jumlah sampel tidak sesuai."

    )


VALIDASI STRUKTUR VISUALISASI


,Komponen,Jumlah
0,Total Sampel,30
1,Prediksi Benar,15
2,Misclassification,10
3,High Uncertainty,5



✅ Struktur visualisasi sesuai dengan desain penelitian.


# BAB 4 Visualisasi SHAP

Bab ini membangun ulang visualisasi SHAP menggunakan model IndoBERT terbaik. Visualisasi mengikuti dokumentasi resmi SHAP sehingga menghasilkan tampilan token berwarna merah muda (kontribusi positif) dan biru (kontribusi negatif).